In [1]:
# 01_data_integrity.ipynb
# Consistency checks and parameter reconciliation for the HITL-AI credit-rating pipeline.

## 1. Load data

In [2]:
import json
import numpy as np
import pandas as pd
DATA = '../data'
tasks = pd.read_csv(f'{DATA}/tasks.csv')
with open(f'{DATA}/queue_params.json') as fh:
    qp = json.load(fh)
tasks

,task_id,task_name,pre_ai_hours,post_ai_hours,verification_hours,adoption_rate,error_severity,accountability_constraint,compression_type
0,T1,Financial/disclosure data compilation,2.5,0.2,0.1,0.95,2,0,high_compression
1,T2,LLM opinion draft & RAG citation,3.0,0.1,0.2,0.80,3,0,high_compression
2,T3,Citation source verification (fact-check),1.5,2.5,2.5,0.40,5,1,partial_or_irreducible
3,T4,Chatbot answer/manual refinement,1.0,0.4,0.3,0.85,2,0,partial
4,T5,System grade monotonicity check,2.0,0.3,0.2,0.90,4,0,high_compression
5,T6,Evaluator override rationale drafting,0.5,1.5,1.5,0.10,5,1,irreducible
6,T7,Governance logging packaging,1.5,0.2,0.1,0.98,3,1,high_compression


## 2. Derived per-task quantities
At the review stage the AI draft replaces the human task in full (generation is machine time), so the
**gross** AI time saving is `g_i = tau_i = pre_ai_hours`. The net value of full delegation is
`net_i = g_i - v_i = tau_i - v_i`, which equals the reduction in review-stage service time in the queueing
layer, `S_i = (1-x_i) tau_i + x_i v_i`. Using `pre - post` as the saving would charge verification twice,
because the observed post-AI time already contains the checking time (e.g. T3: post = v = 2.5 h).

In [3]:
tasks['g_i'] = tasks['pre_ai_hours']                      # gross saving = tau_i
tasks['net_i'] = tasks['g_i'] - tasks['verification_hours']  # = tau_i - v_i
tasks['observed_change'] = tasks['pre_ai_hours'] - tasks['post_ai_hours']
tasks['delegation_loss'] = tasks['net_i'] < 0
tasks[['task_id', 'g_i', 'verification_hours', 'net_i', 'observed_change', 'delegation_loss']]

,task_id,g_i,verification_hours,net_i,observed_change,delegation_loss
0,T1,2.5,0.1,2.4,2.3,False
1,T2,3.0,0.2,2.8,2.9,False
2,T3,1.5,2.5,-1.0,-1.0,True
3,T4,1.0,0.3,0.7,0.6,False
4,T5,2.0,0.2,1.8,1.7,False
5,T6,0.5,1.5,-1.0,-1.0,True
6,T7,1.5,0.1,1.4,1.3,False


## 3. Integrity issue 1 - Queueing parameters
Reported: `E[S]=4.0h`, range `0.5-12h`, prime share `0.60`. Treating `0.5h/12h` as type means is inconsistent;
they are range endpoints. The fixed scenario values are `s_fast=1.33h`, `s_slow=8.0h`, `p=0.60`.

In [4]:
p = qp['client_mix']['prime_share']
s_min, s_max = qp['service_time_min_hours'], qp['service_time_max_hours']
print(f'E[S] if 0.5/12 were means: {p*s_min+(1-p)*s_max:.2f} h  (reported: 4.0 h)')
print('Required prime mean s_fast for E[S]=4.0 (p=0.60):')
for s_slow in [6, 7, 8, 9, 10, 11, 12]:
    s_fast = (4.0 - (1 - p) * s_slow) / p
    print(f'  s_slow={s_slow:>4.1f} -> s_fast={s_fast:>6.2f}  [{"OK" if 0.5 <= s_fast <= s_slow else "infeasible"}]')
tm = qp['service_time_type_means']
sf, ss = tm['s_fast_prime_hours'], tm['s_slow_scarce_hours']
ES = p*sf + (1-p)*ss; ES2 = p*sf**2 + (1-p)*ss**2
print(f'Fixed scenario: s_fast={sf}, s_slow={ss}, p={p} -> E[S]={ES:.3f}, Var={ES2-ES**2:.2f}, CV={np.sqrt(ES2-ES**2)/ES:.3f}')

E[S] if 0.5/12 were means: 5.10 h  (reported: 4.0 h)
Required prime mean s_fast for E[S]=4.0 (p=0.60):
  s_slow= 6.0 -> s_fast=  2.67  [OK]
  s_slow= 7.0 -> s_fast=  2.00  [OK]
  s_slow= 8.0 -> s_fast=  1.33  [OK]
  s_slow= 9.0 -> s_fast=  0.67  [OK]
  s_slow=10.0 -> s_fast=  0.00  [infeasible]
  s_slow=11.0 -> s_fast= -0.67  [infeasible]
  s_slow=12.0 -> s_fast= -1.33  [infeasible]
Fixed scenario: s_fast=1.33, s_slow=8.0, p=0.6 -> E[S]=3.998, Var=10.68, CV=0.817


## 4. Integrity issue 2 - Residual-labour share

In [5]:
irr = tasks[tasks['compression_type'].isin(['irreducible', 'partial_or_irreducible'])]
post_total = tasks['post_ai_hours'].sum(); pre_total = tasks['pre_ai_hours'].sum()
t4_post = tasks.loc[tasks.task_id == 'T4', 'post_ai_hours'].iloc[0]
defs = {
    'D1  irreducible / post-AI human time': 100 * irr['post_ai_hours'].sum() / post_total,
    "D1' + half partial (T4)": 100 * (irr['post_ai_hours'].sum() + 0.5 * t4_post) / post_total,
    'D2  irreducible post / pre-AI total': 100 * irr['post_ai_hours'].sum() / pre_total,
    'D3  interview self-report': 62.5,
    'D4  task count': 100 * len(irr) / len(tasks),
}
for k, val in defs.items():
    print(f'{k:<40} {val:5.1f}%')

D1  irreducible / post-AI human time      76.9%
D1' + half partial (T4)                   80.8%
D2  irreducible post / pre-AI total       33.3%
D3  interview self-report                 62.5%
D4  task count                            28.6%


## 5. Delegation-paradox confirmation

In [6]:
print(tasks[tasks['delegation_loss']][['task_id', 'g_i', 'verification_hours', 'net_i']].to_string(index=False))

task_id  g_i  verification_hours  net_i
     T3  1.5                 2.5   -1.0
     T6  0.5                 1.5   -1.0
